In [2]:
%load_ext autoreload
%autoreload 2

In [9]:
import sys
import pdb
import numpy as np
import polars as pl  
from pathlib import Path
from sklearn.metrics import confusion_matrix,accuracy_score, roc_auc_score

sys.path.append('../')
from NeuroHDC.FHRR import *
from NeuroHDC.fn import *
from conformalHDC.models import *
from conformalHDC.methods import *
from conformalHDC.utils import *

In [13]:
# --------------
# load data
# --------------
# NOTE. scale params, control the width for guassian kernel
# small value makes the code robust to small rate changes big value make class more separate
beta = 0.3
dim = 15000 # 10k-20k
training_window = (200,600)
bin_size = 25
bin_step=2
window_size=200
seed = 0

in_path = Path("../data/rat") / f"odor_prep_{training_window}_{bin_size}.pickle"

cf_list = []

rat_name = ['Barat','Buchanan','Mitt','Stella','Superchris']
for irat in range(0,5):
    splits = prep_loader_slicing(
        irat=irat, 
        split_ratio=(0.5, 0.4, 0.1), 
        in_path=in_path,
        step_bins=bin_step,        # Stride
        slicing_window=window_size, # 200ms window
        bin_size=bin_size,
        seed=seed
    )

    X_train, y_train = splits.train.X, splits.train.y
    X_cal, y_cal = splits.cal.X, splits.cal.y
    X_test, y_test = splits.test.X, splits.test.y
    
    # Build feature Encoder
    n, nT, p = X_train.shape 
    rff = RFF(n_feature=p, dimension=dim, seed=seed)
    W = rff.gen_basis(cov=np.eye(p)) 
    TB = rff.gen_time_base()
    
    # Encode
    encoded_x_train = rff.encode_all(W, X_train, TB, beta=beta)
    encoded_x_cal   = rff.encode_all(W, X_cal, TB, beta=beta)
    encoded_x_test  = rff.encode_all(W, X_test, TB, beta=beta)
    
    # build prototype
    class_proto = rff.build_class_prototypes(encoded_x_train, y_train)
    
    # Optional: Use Iterative Novelty Learning
    # only 1-2 rats get better
    # class_proto = rff.train_iterative_novelty(encoded_x_train, y_train, iterations=5) 
    
    # evaluate one-pass performance
    y_train_pred = rff.decode(encoded_x_train, class_proto)
    y_cal_pred = rff.decode(encoded_x_cal, class_proto)
    y_test_pred = rff.decode(encoded_x_test, class_proto)

    print(f"\nRat {rat_name[irat]} Summary: {summary(splits)}\n")
    print(f"Encoding shape: train {encoded_x_train.shape}, cal {encoded_x_cal.shape}, test {encoded_x_test.shape}\n")
    print(f"  ACC Train: {accuracy_score(y_train, y_train_pred):.3f}\n")
    print(f"  ACC Cal:   {accuracy_score(y_cal, y_cal_pred):.3f}\n")
    print(f"  ACC Test:  {accuracy_score(y_test, y_test_pred):.3f}\n")

Original Shape: (133, 16, 92)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (665, 8, 92)

Rat Barat Summary: {'Train': 'X: (330, 8, 92), 330 samples from 66 trials.', 'Cal': 'X: (265, 8, 92), 265 samples from 53 trials.', 'Test': 'X: (70, 8, 92), 70 samples from 14 trials.'}

Encoding shape: train (330, 15000), cal (265, 15000), test (70, 15000)

  ACC Train: 0.776

  ACC Cal:   0.457

  ACC Test:  0.414

Original Shape: (174, 16, 79)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (870, 8, 79)

Rat Buchanan Summary: {'Train': 'X: (435, 8, 79), 435 samples from 87 trials.', 'Cal': 'X: (345, 8, 79), 345 samples from 69 trials.', 'Test': 'X: (90, 8, 79), 90 samples from 18 trials.'}

Encoding shape: train (435, 15000), cal (345, 15000), test (90, 15000)

  ACC Train: 0.837

  ACC Cal:   0.557

  ACC Test:  0.544

Original Shape: (207, 16, 104)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (1035, 8, 104)

Rat Mitt Summary: {'Train': 'X: (515, 8, 104

In [31]:
irat = 4
ID_CLASSES = [0, 1, 2, 3]

# Using the slicing loader
splits = prep_loader_slicing(
    irat=irat, 
    split_ratio=(0.5, 0.4, 0.1), 
    in_path=in_path, 
    step_bins=bin_step,
    slicing_window=window_size,
    seed=seed
)

X_train_id, y_train_id = splits.train.X, splits.train.y
X_cal_id, y_cal_id = splits.cal.X, splits.cal.y
X_test_id, y_test_id = splits.test.X, splits.test.y

Original Shape: (164, 16, 46)
Applying sliding window of 200 ms (8 bins)
Augmented Shape: (820, 8, 46)


In [32]:
run_window = (0,200)
run_in_path = Path("../data/rat") / f"run_prep_{run_window}_{bin_size}.pickle"

with run_in_path.open("rb") as f:
    run_data = pickle.load(f)

# NO normalization needed, just load raw
X_ood = run_data[irat]['binned_spk'] 

print(f"OOD Data shape (Running): {X_ood.shape}")

if X_ood.shape[1] != X_train_id.shape[1]:
    print(f"WARNING: Mismatch in time bins! ID: {X_train_id.shape[1]}, OOD: {X_ood.shape[1]}")
else:
    print("Time bins match.")

OOD Data shape (Running): (26, 8, 46)
Time bins match.


In [33]:
rff = RFF(n_feature=X_train_id.shape[2], dimension=dim, seed=seed)
W = rff.gen_basis(cov=np.eye(X_train_id.shape[2]))
TB = rff.gen_time_base()

# Encode everything
print("Encoding data...")
enc_train_id = rff.encode_all(W, X_train_id, TB, beta)
enc_cal_id = rff.encode_all(W, X_cal_id, TB, beta) # Using .cal
enc_test_id = rff.encode_all(W, X_test_id, TB, beta)
enc_ood = rff.encode_all(W, X_ood, TB, beta)

# Build Prototypes
proto_dict = rff.build_class_prototypes(enc_train_id, y_train_id)
proto_matrix = np.stack([proto_dict[k] for k in ID_CLASSES])
print("Prototypes built.")

Encoding data...
Prototypes built.


In [34]:
chdc = ConformalHDC(
    class_HVs=proto_matrix, 
    class_labels=ID_CLASSES, 
    sim_measure="complex_cosine"
)

In [35]:
preds_vanilla_idx = chdc.predict(enc_test_id)
preds_vanilla = np.array([ID_CLASSES[i] for i in preds_vanilla_idx])
baseline_acc = accuracy_score(y_test_id, preds_vanilla)
print(f"Baseline ACC: {baseline_acc}.")

Baseline ACC: 0.8823529411764706.


In [37]:
SCORE_TYPES = ['sim', 'ratio', 'discount', "penalized", "inverse_quantile"] 
ALPHA = 0.2
results_list = []

for score_type in SCORE_TYPES:
    print(f"Testing Score Type: {score_type}...")
    
    # Calibrate using the Calibration Set
    chdc.compute_calib_scores(enc_cal_id, y_cal_id, score_type=score_type)
    
    # Task 1: Set-Valued Prediction
    sets = chdc.set_valued_CP(enc_test_id, ALPHA, marginal=True)
    df_res = eval_m_psets(sets, y_test_id)
    marginal_cov = df_res['M-coverage'].item()
    avg_size = df_res['M-size'].item()

    # Task 2: Point-valued Prediction (Efficient)
    preds = chdc.point_valued_CP(enc_test_id, method="efficient")
    point_acc = accuracy_score(y_test_id, preds)
    
    # Task 3: OOD Detection (AUROC)
    p_vals_id  = chdc.get_max_p_value(enc_test_id, marginal=True)
    p_vals_ood = chdc.get_max_p_value(enc_ood, marginal=True)
    
    y_true_roc = np.concatenate([np.ones(len(p_vals_id)), np.zeros(len(p_vals_ood))])
    y_scores_roc = np.concatenate([p_vals_id, p_vals_ood])
    ood_auroc = roc_auc_score(y_true_roc, y_scores_roc)
    
    results_list.append({
        "Method": score_type,
        "Alpha": ALPHA,
        "Baseline_Acc": baseline_acc,
        "Point_Acc": point_acc,
        "Set_Coverage": marginal_cov,
        "Set_Size": avg_size,
        "OOD_AUROC": ood_auroc
    })

# Results Table
final_df = pd.DataFrame(results_list)
print("\nFinal Comparison Results:")
print(final_df.round(4).to_string(index=False))

Testing Score Type: sim...
Testing Score Type: ratio...
Testing Score Type: discount...
Testing Score Type: penalized...
Testing Score Type: inverse_quantile...

Final Comparison Results:
          Method  Alpha  Baseline_Acc  Point_Acc  Set_Coverage  Set_Size  OOD_AUROC
             sim    0.2        0.8824     0.8824        0.9176    3.3529     0.9658
           ratio    0.2        0.8824     0.8824        0.9176    1.1412     0.5529
        discount    0.2        0.8824     0.8824        0.9176    2.8471     0.9760
       penalized    0.2        0.8824     0.8824        0.6471    2.0000     0.0706
inverse_quantile    0.2        0.8824     0.8824        0.9176    1.3529     0.5964
